This script (Tomofast-x-µ) runs Tomofast-x Code for QDM 3D Inversion, Magnetic source location determination, and Magnetic vector direction and intensity Detection

 The full workflow:
 1. Three-Dimensional Vector Magnetization Inversion: joint inversion of the full vector field (Bx, By, Bz), or Bz alone, over a discretized 3D volume, recovering the magnetization vector (Mx, My, Mz) at every cell; suited to complex or non-dipolar sources, and a complete result on its own.
 2. Equivalent Dipole Location: for sources well-approximated as point dipoles, a steepest-descent search locates the single cell of minimum data misfit, initialized from Step 1's max-magnetization cell or the Bz anomaly peaks.
 3. Global Multi-Dipole Inversion: a joint inversion at all identified dipole centers to recover each source's moment, declination, and inclination.

 Every step in an independent tool

 To Run this code you have to install Tomofast-x first, and copy paste the current code in the same folder beside Bz file, this code automatically organizes the outputs within some folders


 Conversion rule between Tomofast and QDM

**QDM:** Z-axis positive *up*  
**Tomofast:** Z-axis positive *down*

| Conversion | Rule |
|-----------|------|
| **QDM → Tomofast** | Flip both **Bz** and **Mz** |
| **Tomofast → QDM** | Flip both **Bz** and **Mz** |
| **Tomofast → Tomofast** | Don’t flip **Bz** and don’t flip **Mz** |
| **QDM → QDM** | Don’t flip **Bz** and don’t flip **Mz** |


# Import Libraries


In [ ]:
### Import Libraries and Settings ###
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import tomofastx_mu as tmu

# printing control: how much is shown for each step (None = show everything)
tmu.PRINT_LIMIT = 30   # max lines printed per step
tmu.PLOT_LIMIT = 10    # max figures shown per step (all figures are still saved)

# Tomofast Directory
TOMOFAST_HOME_WSL = "/home/abekhit/Tomofast-X"
TOMOFAST_HOME = "/home/abekhit/Tomofast-X"

# WSL Directory
WSL = r"C:\Windows\System32\wsl.exe"

# Num of cores for tomofast run
NPROC = 7 # mpirun -np

# where Tomofast-x runs: False = Linux, True = WSL (Windows)
USE_WSL = True

# define cell size
d = 4.4 # dx=dy=dz cubic cells (should match data spacing - pixel size)

base_dir = os.getcwd()  # current working directory

# picks the next free "QDM"/"QDM1"/... folder name for this run
cfg = tmu.Settings(base_dir, d, TOMOFAST_HOME, TOMOFAST_HOME_WSL, WSL, NPROC, USE_WSL)

# Create Folders


In [ ]:
### Create Folders ###
tmu.create_folders(cfg)

# Quick Visualization
the following two lines show the bz vs LED, assuming Bz.mat is in the same folder of this code.

In [ ]:
### Quick Visualization: Show Bz vs LED ###
tmu.show_bz_vs_led(cfg)

In [ ]:
### Quick Visualization: Show Bz overlaid by LED ###
tmu.show_bz_led_overlay(cfg)

# 1: Data Preparation
This tool prepares the data

### 1. Read Bx, By, Bz mat file
This line reads Bx/By/Bz mat file (qdmlab) format and saves it as csv and plot (if your data is csv skip the first step). Only the field components listed in FIELD_KEYS (currently Bx, By, Bz) are read, saved, and plotted, and if a listed field is missing from the file, it's skipped.
* User Attention: Define the .mat file name, define which field components to read in FIELD_KEYS, and define Vmin max for the plot, given it's very important parameter for seeing tiny dipoles. These lines convert the data from Tesla to nanoTesla.

In [ ]:
### Data Preparation: 1. Read Bx, By, Bz mat file ###
# User Attention: define the Bz.mat name
# tmu.FIELD_KEYS = ["Bx", "By", "Bz"]   # field components to look for in the .mat file

data = tmu.load_qdm_mat(os.path.join(base_dir, "Bz.mat"))

# Dataset available in memory
print(data)

# CSV also saved next to the .mat file (Bz.csv)

In [ ]:
### Data Preparation: 1. Read Bx, By, Bz mat file (plots) ###
tmu.plot_components(data)

### 2. Format data into Tomofast-x format [XYZ Bx By Bz] (or [XYZ Bz] if only Bz is present)
This line reads the csv file output from the previous step, or a file defined by the user, and supports both Bz-only data and Bx/By/Bz data together (auto-detected from the number of columns).
* Also supports data clipping by defining the desired cell indices {x_start = 0, x_end = nx, y_start = 0, y_end = ny}
* User Attention: define the CSV file name (filename_z=)
* User Attention: to change the instrument height, adjust z_coords (currently set to the original file's z-values; set to 0 or another constant if using a fixed sensor-sample distance)

In [ ]:
### Data Preparation: 2. Format data into Tomofast-x format ###
# User Attention: define the csv file name
# clipping by cell index: x_end / y_end = None keeps the full scan
tmu.prepare_tomofast_data(cfg, csv_name="Bz.csv",
                          x_start=0, x_end=None,
                          y_start=0, y_end=None)

### 3. Anomalies Detection and Windowing
This line reads the Bz data and splits the anomalies into windows, using a standard deviation-based thresholding approach to suppress unwanted background signals, focus on the sources, and reduce computational cost — also useful for cases where dipoles are very close and data interferes.
* User Attention: the following line needs user adjustment to capture reliable anomalies:
* threshold_factors : a list of threshold multipliers to sweep through, high to low (e.g. `list(np.arange(10, 0.0000, -0.5))`), instead of one fixed value — strong sources get resolved accurately first, weaker ones revealed after without merging into them. Use a one-element list for a single threshold, e.g. `[5]`. Must start high and step down; starting too low merges strong sources' fringes with nearby weaker ones into oversized, inaccurate boxes.
* lock_margin_factor : once a blob is picked, this fraction of its window is locked out at lower thresholds so its fringe isn't re-picked as a fake new source. Larger = safer against duplicates, smaller = better at recovering close real neighbours.
* min_lock_margin : floor (grid cells) for that lock-out, so tiny blobs still get some protection. Keep at 1+.
* pad_x, and pad_y : extend the window size around the anomaly (use to cover the dipole)
* min_blob_cells : this number ignores the noisy small anomalies based of the cell size printed (set 1 to keep everything)
* skip_blobs this number excludes the noisy small anomalies based of the blob number printed (set empty {} keep everything)
* choose_blobs this number selects only the specific anomalies you want to keep, based on the blob number printed (set empty {} keep everything)
* max_blob_cells: this number ignores the noisy huge anomalies based of the cell size printed (set very big number to keep everything e.g, 100000)
* nz_fixed : this number defines the number of depth cells required for the parfile (step 5); should match the mesh model in the next step.
* for the case where positive and negative dipoles detected as two anomlaies use labels, n_blobs = ndi.label(mask_dilated), otherwise use:labels, n_blobs = ndi.label(mask)

In [ ]:
### Data Preparation: 3. Anomalies Detection and Windowing ###
tmu.detect_anomalies(
    cfg,
    threshold_factors=list(np.arange(10, 0.0000, -0.5)),   # sweep instead of one fixed value, e.g. [5] for a single threshold
    lock_margin_factor=0.4,    # once a blob is picked, lock out this fraction of its own size so a lower threshold can't re-pick it
    min_lock_margin=0,
    pad_x=5,                   # extend rectangle in X (cols)
    pad_y=5,                   # extend rectangle in Y (rows)
    skip_blobs=(),             # ignore noisy blobs (empty = keep everything)
    choose_blobs=(1,10),           # if not empty, ONLY keep these blob IDs
    min_blob_cells=1,          # ignore tiny blobs (1 keeps everything)
    max_blob_cells=100000,     # ignore huge blobs
    nz_fixed=10,               # nz for Tomofast (same for all blobs)
    merge_close=True,          # True = nearby anomalies join into one blob, False = keep them separate
)

# Tool 1: Three-Dimensional Vector Magnetization Inversion of Stray Magnetic Fields
* If you do not wish to perform the full 3D inversion and instead want to proceed directly with dipole locating and fitting, skip it and go to Tool 3.
* This tool runs Tomofast-x full magnetic vector inversion in order to obtain the full 3D magnetic image of the QDM micromagnetic scan.  
the inputs are (Bz file, mesh file, Tomofast Parameter file)

### 1. Create Meshs
this line creates 3d Mesh model.  
this line supports non-uniform padding regions dx , dy (delta per layer), and dz (delta per layer) to reach maximum pads and depth with less computational resources)
* User Attention: define the model parameters:
* dx_core, dy_core: the x,y cell size in the core area below the anomlaies (must match the data spacing #d#)
* npad_x, npad_y : Number of padding cell (outside the windows to ensure full 3D model recovery and overcome the edge effect)
* rx, ry : Padding growth (for example 1.15 means every cell is cummlatively 15% larger than the previous one)
* nz, dz0, rz: number of depth cells, depth cell size, and depth cell growth ( for example 60, 4.4, 1 mean 60 x 4.4 x 1 = 264 µm depth); non-uniform depth saves time and cost for 3d modelling, however, it is not recommended for dipole information as it might lead to non-accurate results. in all cases total depth should matches the sample thickness)

In [ ]:
### Tool 1: 1. Create Meshs ###
tmu.create_meshgrids(
    cfg,
    npad_x=1, npad_y=1,        # pad cells on EACH side in X / Y
    rx=1.05, ry=1.05,          # pad growth factor in X / Y (outwards)
    z_min=0.0,                 # top of model (positive down)
    nz=30,                     # number of z cells
    dz0=None,                  # top layer thickness (um), None = d
    rz=1,                      # z growth factor per layer (depth)
    expand_half_cell=True,     # expand each window by half a core-cell
)

### 2. Create Parfiles
this line creates the parameter files for each anomaly required for the Tomofast-x 3D full vector inversions.
this line reads the number of data [Ndata], mesh sizes [nx,ny,nz] from the anomalies output.
* Compression rate (forward.matrixCompression.rate) is now computed automatically per blob using calc_ideal_compression_rate(), targeting a 1% compression error (Bruce et al. 2026).
* User Attention: set forward.data.magn.nDataComponents = 3 if using Bx, By, Bz, or = 1 if using Bz only.
* User Attention: when using depth weighting set forward.depthWeighting.magn.Z0 to the instrument height, matching Z column in your data.

In [ ]:
### Tool 1: 2. Create Parfiles ###
# n_data_components: 3 if using Bx, By, Bz, or 1 if using Bz only
# z0: depth weighting Z0, the instrument height (matching the Z column in your data)
tmu.create_inversion_parfiles(cfg, n_data_components=3, z0=10.0)

### 3. Run Tomofast-x 3D full vector inversion
this line runs tomofast inversion using the mag.obs, mesh and parfile per window. 
it's also handles transition between Linux (WSL) and windows [automatic copy inputs and outputs)
* WSL or Linux: set USE_WSL in the settings cell at the top

In [ ]:
### Tool 1: 3. Run Tomofast-x 3D full vector inversion ###
tmu.run_3d_inversions(cfg)

### 4. Models Misfits

Reads and plots observed data, calculated data, and residuals

In [ ]:
### Tool 1: 4. Models Misfits ###
tmu.plot_blob_misfits(cfg)

In [ ]:
### Tool 1: 4. Models Misfits (all blobs stitched) ###
tmu.plot_combined_misfit(cfg)

In [ ]:
### Tool 1: 4. Final Data Costs ###
tmu.print_final_data_costs(cfg)

### 5. Extracting Magnetic Volumes from the Inverted Model

this line uses scipy yo detect the 3d grains models based on cell connectivity and amplitude thresholds, calculate mean results per grain, and save 3d plots

In [ ]:
### Tool 1: 5. Extracting Magnetic Volumes from the Inverted Model ###
tmu.extract_grains(cfg)

In [ ]:
### Tool 1: 5. Extracting Magnetic Volumes (largest grains only) ###
# same as above but this keeps the largest grain per blob, useful for filtering compression artifacts
tmu.extract_grains(cfg, keep_largest=True)

In [ ]:
### Tool 1: 5. 3D grains plots ###
tmu.plot_grains_3d(cfg)

### 6. 3D Calculations
Computes per-grain magnetic direction and moment statistics from the voxel-level `grain_labels_{threshold}.csv` files produced by grain detection, looping over every blob and every threshold level (0.1–0.9).

For each grain, the script computes:
- **Mean magnetization vector** (`Mx_mean`, `My_mean`, `Mz_mean`), with the Mz sign flipped to match the Tomofast -> QDM (z-down -> z-up) convention.
- **Dec/Inc** of the mean direction.
- **Resultant ratio (R)** and **angular standard deviation**, a Fisher-style measure of how tightly the individual voxels within the grain agree in direction (R -> 1 = coherent, R -> 0 = scattered).
- **Net moment amplitude** - the *sum* of all voxel magnitudes in the grain, multiplied by a fixed, user-defined cell volume `V`.
- **Moment-weighted dipole position** (`Dipole_X/Y/Z`) - the magnetization-weighted centroid of the grain, used as an initial guide for later dipole-location searches.
* **User attention:** set the cell size (`dx, dy, dz`) at the top of the script to match your model's actual voxel dimensions - this fixed value is used for every grain's moment and dipole-position calculation.
* Results are saved per blob, per threshold, as `grain_dec_inc_{threshold}.csv` in each blob's `Grains_detection` folder.

In [ ]:
### Tool 1: 6. 3D Calculations ###
# this is the mean per grain, not the dipole result
# User Attention: the cell size d (settings cell) sets the volume used for the moment
tmu.grain_statistics(cfg)

In [ ]:
### Tool 1: 6. 3D Calculations (largest grains only) ###
# same as above but also keeps the largest grain per blob (filters errors)
tmu.grain_statistics(cfg, keep_largest=True)

In [ ]:
### Tool 1: 6. Plot Dec, Inc, Moment vs Threshold (largest grains) ###
# one line per blob (reads grain_dec_inc_largest_ALL.csv)
tmu.plot_dec_inc_moment_largest(cfg)

In [ ]:
### Tool 1: 6. Plot Dec, Inc, Moment vs Threshold ###
# one line per blob (reads grain_dec_inc_{threshold}.csv)
tmu.plot_dec_inc_moment(cfg)

### 7. Maximum cell Extraction
this line detects the 3d location of the maximum cell per grain (it will be used as a guide for the source location searching)
* User attention: we assume here the best amplitude threshold is 0.9 (change THR = 0.9 if wanted)
* the output maxi cells shouldn't be very deep; if so increase the padding to make sure the model is fully covered.

In [ ]:
### Tool 1: 7. Maximum cell Extraction ###
tmu.extract_max_cells(cfg)

In [ ]:
### Tool 1: 7. Combine all max cells ###
# combines all the max cells in one file for the source location searching
# User attention: we assume here the best amplitude threshold is 0.9 (change thr if wanted)
df = tmu.combine_max_cells(cfg, thr=0.9)

In [ ]:
### Tool 1: 7. Max cells plot (coloured by depth) ###
tmu.plot_max_cells_depth(cfg, thr=0.9)

# Tool 2: Locating Equivalent Dipole Centers
This tool runs a series of single cell Tomofast-x full magnetic vector inversion in order to obtain the true source location which matches the minimum data cost(misfit). 
* if you are starting from here run Tool 1 lines [1-3].

### 1. Create Parfile
this line creates parfiles for the single cell inversions.
* User Attention: set forward.data.magn.nDataComponents = 3 if using Bx, By, Bz , or = 1 if using Bz only.
* User Attention: when using depth weighting set forward.depthWeighting.magn.Z0 to the instrument height, matching Z column in your data.

In [ ]:
### Tool 2: 1. Create Parfile ###
# n_data_components: 3 if using Bx, By, Bz, or 1 if using Bz only
# z0: depth weighting Z0, the instrument height (matching the Z column in your data)
tmu.create_single_cell_parfiles(cfg, n_data_components=3, z0=10.0)

### 2. Run Tomofast for minimum Data Cost Detection
This script performs a steepest-descent search for each detected blob to find the single-cell position that minimizes the Tomofast-x data cost, starting either from 3D inversion maximum cells (`maxs`) or anomaly seed centers (`seed`).
* WSL or Linux: set USE_WSL in the settings cell at the top
* thr : define the threshold of the maximums that matches the model (e.g, one grain per one dipole)
* start_mode = choose  "seed" or "maxs

In [ ]:
### Tool 2: 2. Run Tomofast for minimum Data Cost Detection ###
# start_mode: "seed" = anomaly centers, "maxs" = max cells of the 3D inversion
# thr: threshold of the max cells (used with "maxs")
best_per_blob = tmu.run_steepest_descent(cfg, start_mode="seed", thr=0.9, max_steps=200)

In [ ]:
### Tool 2: Clean grains ###
# remove duplicate picks, grains that hit the last step, grains outside the map and grains with a high data cost
# run this AFTER the steepest-descent cell, BEFORE the global inversion
tmu.clean_grains(
    cfg,
    max_step_to_remove=1000,   # remove grains whose best step == this number (the run's max_steps); a bigger number removes none
    cost_threshold=1,          # keep only grains at or below this data cost (0, 1)
    bz_file=None,              # or the Bz file to read the map extent from, e.g. mag_observed.txt
    x_map=(0, 2079),           # map (Bz) extent in um, check against your map
    y_map=(0, 1287),
    z_map=(0, 10000),          # model z range in the same sign convention as the mesh files; None turns a side off
)

### 3. Analysis Plots

In [ ]:
### Tool 2: 3. Analysis Plots ###
tmu.plot_blob_costs(cfg)

# Tool 3: Global Multi-Dipole Inversion
this tool fits the full data with the detected true sources locations in a single inversion run

### 1. Combine best locations 
(into one mesh file)

In [ ]:
### Tool 3: 1. Combine best locations ###
# best location per anomaly (i from X, j=1, k=1)
# has_header=True if best_per_blob_steepest.txt still has its header line, skip_nans=True to skip blobs without a valid cost
n_blobs = tmu.combine_best_meshes(cfg)

### 2. Create Parfile
create parfile for data fitting in a single run inversion.
* User Attention: set forward.data.magn.nDataComponents = 3 if using Bx, By, Bz, or = 1 if using Bz only.
* User Attention: when using depth weighting set forward.depthWeighting.magn.Z0 to the instrument height, matching Z column in your data.

In [ ]:
### Tool 3: 2. Create Parfile ###
# n_data_components: 3 if using Bx, By, Bz, or 1 if using Bz only
# z0: depth weighting Z0, the instrument height (matching the Z column in your data)
tmu.create_dipole_parfile(cfg, n_blobs, n_data_components=3, z0=10.0)

### 3. Run Tomofast inversion for Dipole Fitting
fit the true detected locations with the full data
* WSL or Linux: set USE_WSL in the settings cell at the top

In [ ]:
### Tool 3: 3. Run Tomofast inversion for Dipole Fitting ###
tmu.run_dipole_inversion(cfg)

### 4. Vectors Directions Analysis

In [ ]:
### Tool 3: 4. Stereonet plot (combined grains) ###
tmu.plot_stereonet(cfg)

In [ ]:
### Tool 3: 4. Stereonet plot (combined grains), color by moment ###
tmu.plot_stereonet_by_moment(cfg)

### 5. Final Report and plots

In [ ]:
### Tool 3: 5. Final Report and plots ###
# position, direction and moment of every dipole (one model, many blob locations)
# options: has_header=True, skip_nans=True (skip blobs without a valid cost),
#          cell_size_from_bounds=True (dx, dy, dz from each blob's bounds, for a non-uniform model)
results_table = tmu.build_final_report(cfg)

In [ ]:
### Tool 3: 5. Combined figure: Bz, centres by depth, declination arrows by moment ###
tmu.plot_combined_figure(cfg)

In [ ]:
### Tool 3: 5. Plot dipole locations on LED on Bz ###
tmu.plot_centres_on_led(cfg)

In [ ]:
### Tool 3: 5. Plot obs, calc, misfit (Bx, By, Bz if present, else Bz only) ###
tmu.plot_obs_calc_misfit(cfg)

# Analysis: Compare the QDM grain scale results to the SQUID bulk sample scale 
* User Attention: define the bulk measuremnts dec, inc, moment

In [ ]:
### Analysis: Compare the QDM grain scale results to the SQUID bulk sample scale ###
# User Attention: define the bulk measurement dec, inc, moment
tmu.compare_mp_mf(
    cfg,
    irm_dec=270,               # degrees
    irm_inc=0.5,               # degrees
    irm_moment_am2=1.94e-5,    # A*m^2 (bulk sample net moment)
    mf_threshold=0.91,         # fixed threshold used for the m^f comparison
    bz_vmax=4000,              # colour limit of the Bz maps (nT)
)